# tau-chrono Quickstart: Bayesian Noise Tracking for Quantum Circuits

This notebook demonstrates how **tau-chrono** tracks noise through quantum circuits
using Bayesian-updated Petz recovery maps -- no quantum hardware or Qiskit needed.

**What you will learn:**
1. How to create standard noise channels (depolarizing, amplitude damping)
2. How the **tau parameter** quantifies recovery failure
3. How **Bayesian composition** outperforms naive noise estimation
4. How to visualize depth scaling

**Requirements:** `numpy`, `matplotlib` (both standard; no quantum SDK needed)

## Cell 1: Install and Import

In [ ]:
import sys, os
import numpy as np

# Add the repo root to the path so we can import tau_chrono
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from tau_chrono import (
    depolarizing,
    amplitude_damping,
    dephasing,
    verify_cptp,
    apply_channel,
    fidelity,
    tau_parameter,
    bayesian_compose,
    compose_kraus,
)

print(f"numpy version:      {np.__version__}")
print(f"tau_chrono imported successfully")
print(f"Ready to go!")

## Cell 2: Create Noise Channels

We build three standard noise channels and verify they are CPTP
(completely positive, trace-preserving):

- **Depolarizing**: uniform Pauli errors with probability *p*
- **Amplitude damping**: energy relaxation (T1 decay) with probability *gamma*
- **Dephasing**: phase randomization (T2 decay) with probability *p*

In [ ]:
# Create noise channels
depol_kraus   = depolarizing(p=0.05)       # 5% depolarizing noise
ad_kraus      = amplitude_damping(gamma=0.08)  # 8% amplitude damping
deph_kraus    = dephasing(p=0.04)          # 4% dephasing

# Verify CPTP
channels = {
    "Depolarizing (p=0.05)":       depol_kraus,
    "Amplitude damping (g=0.08)":  ad_kraus,
    "Dephasing (p=0.04)":          deph_kraus,
}

print(f"{'Channel':<35s}  {'# Kraus ops':>12s}  {'CPTP':>6s}")
print("-" * 58)
for name, kraus in channels.items():
    ok = verify_cptp(kraus)
    print(f"{name:<35s}  {len(kraus):>12d}  {'PASS' if ok else 'FAIL':>6s}")

# Show a Kraus operator
print("\nDepolarizing K_0 (identity-like):")
print(np.round(depol_kraus[0], 4))

## Cell 3: Run Bayesian Composition on a 5-Gate Circuit

We simulate a 5-gate circuit where each gate introduces depolarizing noise.
The **Bayesian composition engine** tracks how sigma (the reference state) evolves
through each gate, giving per-gate tau estimates that account for accumulated noise.

The key parameter **tau** measures recovery failure:
- `tau = 0`: perfect recovery (the channel is reversible)
- `tau = 1`: complete failure (information fully lost)

In [ ]:
# Build a 5-gate circuit with mixed noise
circuit_channels = [
    depolarizing(0.03),        # Gate 0: light depolarizing
    amplitude_damping(0.05),   # Gate 1: amplitude damping
    depolarizing(0.04),        # Gate 2: moderate depolarizing
    dephasing(0.06),           # Gate 3: dephasing
    depolarizing(0.03),        # Gate 4: light depolarizing
]
gate_names = ["Depol(0.03)", "AD(0.05)", "Depol(0.04)", "Deph(0.06)", "Depol(0.03)"]

# Initial states
sigma_0 = np.eye(2, dtype=complex) / 2   # maximally mixed
rho_0 = np.array([[0.5, 0.5],
                   [0.5, 0.5]], dtype=complex)  # |+> state

# Run Bayesian composition
result = bayesian_compose(
    channels=circuit_channels,
    sigma_0=sigma_0,
    rho=rho_0,
    channel_names=gate_names,
)

# Display per-gate results
print(f"{'Gate':<14s}  {'tau_naive':>10s}  {'tau_eff':>10s}  {'Class':>16s}")
print("-" * 56)
for gr in result.gate_results:
    print(f"{gr.channel_name:<14s}  {gr.tau_naive:10.6f}  {gr.tau_eff:10.6f}  {gr.classification:>16s}")

print(f"\n--- Totals ---")
print(f"  Bayesian tau (actual):       {result.tau_bayesian_total:.6f}")
print(f"  Multiplicative tau (naive):  {result.tau_multiplicative_total:.6f}")
print(f"  Improvement:                 {result.improvement_percent:.1f}%")
print(f"  Composition inequality:      {'HOLDS' if result.composition_holds else 'VIOLATED'}")
print(f"  Slack:                       {result.composition_slack:.6f}")

## Cell 4: Improvement Table (Naive vs Bayesian)

We sweep circuit depth from 1 to 20 gates and compare the naive multiplicative
estimate (which assumes independent gates) against the actual Bayesian-composed
tau. The improvement grows with depth because the Bayesian tracker correctly
accounts for sigma convergence.

In [ ]:
# Depth sweep: naive vs Bayesian tau
p_noise = 0.05
sigma_0 = np.eye(2, dtype=complex) / 2
rho_0 = np.array([[0.5, 0.5], [0.5, 0.5]], dtype=complex)

depths = list(range(1, 21))
tau_naive_list = []
tau_bayesian_list = []
improvement_list = []

for depth in depths:
    ch = [depolarizing(p_noise) for _ in range(depth)]
    res = bayesian_compose(ch, sigma_0, rho_0)
    tau_naive_list.append(res.tau_multiplicative_total)
    tau_bayesian_list.append(res.tau_bayesian_total)
    improvement_list.append(res.improvement_percent)

# Print table
print(f"{'Depth':>6s}  {'tau_naive':>10s}  {'tau_bayesian':>13s}  {'Improvement':>12s}")
print("-" * 46)
for i, d in enumerate(depths):
    print(f"{d:>6d}  {tau_naive_list[i]:10.6f}  {tau_bayesian_list[i]:13.6f}  {improvement_list[i]:11.1f}%")

print(f"\nPeak improvement: {max(improvement_list):.1f}% at depth {depths[np.argmax(improvement_list)]}")

## Cell 5: Depth Scaling Plot

Visualize how tau grows with circuit depth for both the naive (multiplicative)
and Bayesian estimates. The gap between them is the **value of noise tracking**.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: tau vs depth ---
ax1.plot(depths, tau_naive_list, 'rs-', linewidth=2, markersize=5,
         label='Naive (multiplicative)')
ax1.plot(depths, tau_bayesian_list, 'go-', linewidth=2, markersize=5,
         label='Bayesian (actual)')
ax1.fill_between(depths, tau_bayesian_list, tau_naive_list,
                  alpha=0.15, color='green', label='Improvement region')
ax1.set_xlabel('Circuit depth (number of gates)', fontsize=12)
ax1.set_ylabel(r'Total $\tau$ (recovery failure)', fontsize=12)
ax1.set_title(r'$\tau$ vs Circuit Depth (depolarizing $p=0.05$)', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, 20)
ax1.set_ylim(0, 1.05)

# --- Right panel: improvement % ---
ax2.bar(depths, improvement_list, color='#2ecc71', edgecolor='#27ae60', alpha=0.8)
ax2.set_xlabel('Circuit depth (number of gates)', fontsize=12)
ax2.set_ylabel('Improvement over naive (%)', fontsize=12)
ax2.set_title('Bayesian Improvement Over Naive Estimate', fontsize=13)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xlim(0.5, 20.5)

plt.tight_layout()
plt.show()

## Cell 6: Try It Yourself

Change the noise parameters below and re-run to see how different noise
profiles affect the Bayesian improvement.

**Things to try:**
- Increase `p_noise` to 0.1 or 0.2 -- the improvement percentage changes
- Switch to `amplitude_damping` or `dephasing` channels
- Mix different channel types in the circuit
- Change `max_depth` to see deeper circuits

In [ ]:
# ============================================================
# CHANGE THESE PARAMETERS
# ============================================================
noise_type = "depolarizing"   # "depolarizing", "amplitude_damping", or "dephasing"
p_noise = 0.05                # noise strength (0 to 1)
max_depth = 15                # maximum circuit depth to test
# ============================================================

sigma_0 = np.eye(2, dtype=complex) / 2
rho_0 = np.array([[0.5, 0.5], [0.5, 0.5]], dtype=complex)

# Channel factory
def make_channel(noise_type, p):
    if noise_type == "depolarizing":
        return depolarizing(p)
    elif noise_type == "amplitude_damping":
        return amplitude_damping(p)
    elif noise_type == "dephasing":
        return dephasing(p)
    else:
        raise ValueError(f"Unknown noise type: {noise_type}")

depths = list(range(1, max_depth + 1))
results_custom = []

for depth in depths:
    ch = [make_channel(noise_type, p_noise) for _ in range(depth)]
    res = bayesian_compose(ch, sigma_0, rho_0)
    results_custom.append(res)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
tau_n = [r.tau_multiplicative_total for r in results_custom]
tau_b = [r.tau_bayesian_total for r in results_custom]
imp = [r.improvement_percent for r in results_custom]

ax.plot(depths, tau_n, 'rs-', lw=2, ms=5, label='Naive')
ax.plot(depths, tau_b, 'go-', lw=2, ms=5, label='Bayesian')
ax.fill_between(depths, tau_b, tau_n, alpha=0.15, color='green')
ax.set_xlabel('Depth', fontsize=12)
ax.set_ylabel(r'$\tau$', fontsize=14)
ax.set_title(f'{noise_type} (p={p_noise}): Bayesian vs Naive', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Summary
peak_imp = max(imp)
peak_d = depths[np.argmax(imp)]
print(f"\nNoise: {noise_type}, p={p_noise}")
print(f"Peak improvement: {peak_imp:.1f}% at depth {peak_d}")
print(f"Final tau (depth {max_depth}): naive={tau_n[-1]:.4f}, bayesian={tau_b[-1]:.4f}")